# Fusion des datasets de liste des exercices

## Imports

In [21]:
import kagglehub
from pathlib import Path
import pandas as pd
from pathlib import Path
import datetime

## Téléchargement KaggleHub
Ce bloc télécharge les trois datasets et retrouve automatiquement leurs CSV

In [3]:
def load_first_csv_from_kagglehub(dataset_name: str) -> pd.DataFrame:
    """
    Télécharge un dataset via kagglehub et charge automatiquement
    le premier CSV trouvé dans le dossier.
    """
    path = Path(kagglehub.dataset_download(dataset_name))
    csv_files = list(path.rglob("*.csv"))
    
    if not csv_files:
        raise FileNotFoundError(f"Aucun CSV trouvé dans {path}")
    
    print(f"📥 Dataset '{dataset_name}' chargé depuis : {path}")
    print(f"→ CSV utilisé : {csv_files[0].name}")
    
    return pd.read_csv(csv_files[0]), csv_files[0]

# --- Téléchargement des trois datasets Kaggle ---
df_gym1, file1 = load_first_csv_from_kagglehub("niharika41298/gym-exercise-data")
df_gym2, file2 = load_first_csv_from_kagglehub("rishitmurarka/gym-exercises-dataset")
df_best50, file3 = load_first_csv_from_kagglehub("prajwaldongre/best-50-exercise-for-your-body")

print("Gym Exercise Data:", df_gym1.shape)
print("Gym Exercises Dataset:", df_gym2.shape)
print("Best 50 exercises:", df_best50.shape)

display(df_gym1.head(3))
display(df_gym2.head(3))
display(df_best50.head(3))

📥 Dataset 'niharika41298/gym-exercise-data' chargé depuis : C:\Users\fback\.cache\kagglehub\datasets\niharika41298\gym-exercise-data\versions\1
→ CSV utilisé : megaGymDataset.csv
📥 Dataset 'rishitmurarka/gym-exercises-dataset' chargé depuis : C:\Users\fback\.cache\kagglehub\datasets\rishitmurarka\gym-exercises-dataset\versions\1
→ CSV utilisé : gym_exercise_dataset.csv
📥 Dataset 'prajwaldongre/best-50-exercise-for-your-body' chargé depuis : C:\Users\fback\.cache\kagglehub\datasets\prajwaldongre\best-50-exercise-for-your-body\versions\1
→ CSV utilisé : Top 50 Excerice for your body.csv
Gym Exercise Data: (2918, 9)
Gym Exercises Dataset: (617, 17)
Best 50 exercises: (50, 8)


,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


,Exercise Name,Equipment,Variation,Utility,Mechanics,Force,Preparation,Execution,Target_Muscles,Synergist_Muscles,Stabilizer_Muscles,Antagonist_Muscles,Dynamic_Stabilizer_Muscles,Main_muscle,Difficulty (1-5),Secondary Muscles,parent_id
0,Neck Flexion,Cable,No,Basic or Auxiliary,Isolated,Pull,Sit on bench facing away from middle pulley. P...,Move head away from pulley by bending neck for...,"Sternocleidomastoid,","None,","Rectus Abdominis, Obliques,",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
1,Neck Flexion,Lever (plate loaded),No,Basic or Auxiliary,Isolated,Pull,Sit on seat in machine. Position padded lever ...,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,","None,","Latissimus Dorsi, Deltoid, Posterior, Rhomboid...",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
2,Lateral Neck Flexion,Lever (plate loaded),No,Auxiliary,Isolated,Pull,Sit on seat in machine with feet apart . Pos...,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,","Splenius, Erector Spinae, Levator Scapulae, Tr...","Latissimus Dorsi, Pectoralis Major, Sternal, P...",NaN,NaN,Neck,2,"Sternocleidomastoid, Levator Scapulae",NaN


,Name of Exercise,Sets,Reps,Benefit,Burns Calories (per 30 min),Target Muscle Group,Equipment Needed,Difficulty Level
0,Push-ups,3,15,Builds upper body strength,200,"Chest, Triceps, Shoulders",NaN,Intermediate
1,Squats,4,12,Strengthens lower body,223,"Quadriceps, Hamstrings, Glutes",NaN,Beginner
2,Lunges,3,10,Improves balance and coordination,275,"Quadriceps, Hamstrings, Glutes",NaN,Beginner


## Harmoniser les colonnes (mapping de la capture Nicolas)

On se crée un schéma commun

In [4]:
CANONICAL_COLS = [
    "exercise_name",       # Nom de l'exercice
    "execution",           # Description / Execution
    "target_muscles",      # BodyPart / Target_Muscles / Target muscle Group
    "equipment",           # Equipment / Equipment needed
    "difficulty",          # Level / Difficulty level
    "source_dataset",      # d'où vient la ligne
]


### Dataset Gym_Exercise_DataSet (niharika41298)

D’après la capture :
- Nom de l'exo → nom
- Description → execution
- BodyPart → target_muscles
- Equipment → equipment
- Level → difficulty

In [10]:
df1 = pd.DataFrame({
    "exercise_name":  df_gym1["Title"],
    "execution":      df_gym1["Desc"],
    "target_muscles": df_gym1["BodyPart"],
    "equipment":      df_gym1["Equipment"],
    "difficulty":     df_gym1["Level"],
})
df1["source_dataset"] = "gym_exercise_data"
df1.head()

,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Partner plank band row,The partner plank band row is an abdominal exe...,Abdominals,Bands,Intermediate,gym_exercise_data
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Abdominals,Bands,Intermediate,gym_exercise_data
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Abdominals,Bands,Intermediate,gym_exercise_data
3,Banded crunch,The banded crunch is an exercise targeting the...,Abdominals,Bands,Intermediate,gym_exercise_data
4,Crunch,The crunch is a popular core exercise targetin...,Abdominals,Bands,Intermediate,gym_exercise_data


### Dataset Gym Exercices (rishitmurarka)

D’après la capture :

- Nom de l'exercice → nom
- Execution → execution
- Target_Muscles → target_muscles
- Equipement → equipment

(s’il y a Difficulty ou Level, on le branche sur difficulty)

In [13]:
df2 = pd.DataFrame({
    "exercise_name":  df_gym2["Exercise Name"],
    "execution":      df_gym2["Execution"],
    "target_muscles": df_gym2["Target_Muscles"],
    "equipment":      df_gym2["Equipment"],
    # adapte le nom exact de la colonne de niveau :
    "difficulty":     df_gym2.get("Difficulty", pd.NA),
})
df2["source_dataset"] = "gym_exercises_dataset"
df2.head()

,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Neck Flexion,Move head away from pulley by bending neck for...,"Sternocleidomastoid,",Cable,<NA>,gym_exercises_dataset
1,Neck Flexion,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,",Lever (plate loaded),<NA>,gym_exercises_dataset
2,Lateral Neck Flexion,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,",Lever (plate loaded),<NA>,gym_exercises_dataset
3,Neck Flexion,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,",Lever (selectorized),<NA>,gym_exercises_dataset
4,Lateral Neck Flexion,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,",Lever (selectorized),<NA>,gym_exercises_dataset


### Best 50 exercises for your body

D’après la description Kaggle : Name of Exercise, Target muscle Group, Equipment needed, Difficulty level.
Kaggle

In [17]:
df3 = pd.DataFrame({
    "exercise_name":  df_best50["Name of Exercise"],
    # ce dataset n'a peut-être pas de vraie description → NaN
    "execution":      pd.NA,
    "target_muscles": df_best50["Target Muscle Group"],
    "equipment":      df_best50["Equipment Needed"],
    "difficulty":     df_best50["Difficulty Level"],
})
df3["source_dataset"] = "best_50_exercises"
df3.head()

,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Push-ups,<NA>,"Chest, Triceps, Shoulders",NaN,Intermediate,best_50_exercises
1,Squats,<NA>,"Quadriceps, Hamstrings, Glutes",NaN,Beginner,best_50_exercises
2,Lunges,<NA>,"Quadriceps, Hamstrings, Glutes",NaN,Beginner,best_50_exercises
3,Burpees,<NA>,Full Body,NaN,Advanced,best_50_exercises
4,Mountain Climbers,<NA>,"Core, Shoulders, Legs",NaN,Intermediate,best_50_exercises


## Fusion + nettoyage léger

In [18]:
df_all = pd.concat([df1, df2, df3], ignore_index=True)

# Optionnel : trim espaces, mettre en lower pour certains champs
for col in ["exercise_name", "target_muscles", "equipment", "difficulty"]:
    df_all[col] = (
        df_all[col]
        .astype("string")
        .str.strip()
    )

# Optionnel : supprimer les doublons sur nom + muscles
df_all_unique = df_all.drop_duplicates(
    subset=["exercise_name", "target_muscles", "equipment"],
    keep="first"
).reset_index(drop=True)

print("Shape fusion brute :", df_all.shape)
print("Shape après dédoublonnage :", df_all_unique.shape)

df_all_unique.head(20)

Shape fusion brute : (3585, 6)
Shape après dédoublonnage : (3576, 6)


,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Partner plank band row,The partner plank band row is an abdominal exe...,Abdominals,Bands,Intermediate,gym_exercise_data
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Abdominals,Bands,Intermediate,gym_exercise_data
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Abdominals,Bands,Intermediate,gym_exercise_data
3,Banded crunch,The banded crunch is an exercise targeting the...,Abdominals,Bands,Intermediate,gym_exercise_data
4,Crunch,The crunch is a popular core exercise targetin...,Abdominals,Bands,Intermediate,gym_exercise_data
5,Decline band press sit-up,The decline band press sit-up is a weighted co...,Abdominals,Bands,Intermediate,gym_exercise_data
6,FYR2 Banded Frog Pump,NaN,Abdominals,Bands,Intermediate,gym_exercise_data
7,Band low-to-high twist,The band low-to-high twist is a core exercise ...,Abdominals,Bands,Intermediate,gym_exercise_data
8,Barbell roll-out,The barbell roll-out is an abdominal exercise ...,Abdominals,Barbell,Intermediate,gym_exercise_data
9,Barbell Ab Rollout - On Knees,The barbell roll-out is an abdominal exercise ...,Abdominals,Barbell,Intermediate,gym_exercise_data


## Fusion + nettoyage léger

In [19]:
df_all = pd.concat([df1, df2, df3], ignore_index=True)

# Nettoyages simples
for col in ["exercise_name", "target_muscles", "equipment", "difficulty"]:
    df_all[col] = df_all[col].astype("string").str.strip()

df_all.head(20)


,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Partner plank band row,The partner plank band row is an abdominal exe...,Abdominals,Bands,Intermediate,gym_exercise_data
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Abdominals,Bands,Intermediate,gym_exercise_data
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Abdominals,Bands,Intermediate,gym_exercise_data
3,Banded crunch,The banded crunch is an exercise targeting the...,Abdominals,Bands,Intermediate,gym_exercise_data
4,Crunch,The crunch is a popular core exercise targetin...,Abdominals,Bands,Intermediate,gym_exercise_data
5,Decline band press sit-up,The decline band press sit-up is a weighted co...,Abdominals,Bands,Intermediate,gym_exercise_data
6,FYR2 Banded Frog Pump,NaN,Abdominals,Bands,Intermediate,gym_exercise_data
7,Band low-to-high twist,The band low-to-high twist is a core exercise ...,Abdominals,Bands,Intermediate,gym_exercise_data
8,Barbell roll-out,The barbell roll-out is an abdominal exercise ...,Abdominals,Barbell,Intermediate,gym_exercise_data
9,Barbell Ab Rollout - On Knees,The barbell roll-out is an abdominal exercise ...,Abdominals,Barbell,Intermediate,gym_exercise_data


## Optionnel : enlever les doublons

In [20]:
df_all_unique = df_all.drop_duplicates(
    subset=["exercise_name", "target_muscles", "equipment"],
    keep="first"
).reset_index(drop=True)

df_all_unique.head(20)


,exercise_name,execution,target_muscles,equipment,difficulty,source_dataset
0,Partner plank band row,The partner plank band row is an abdominal exe...,Abdominals,Bands,Intermediate,gym_exercise_data
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Abdominals,Bands,Intermediate,gym_exercise_data
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Abdominals,Bands,Intermediate,gym_exercise_data
3,Banded crunch,The banded crunch is an exercise targeting the...,Abdominals,Bands,Intermediate,gym_exercise_data
4,Crunch,The crunch is a popular core exercise targetin...,Abdominals,Bands,Intermediate,gym_exercise_data
5,Decline band press sit-up,The decline band press sit-up is a weighted co...,Abdominals,Bands,Intermediate,gym_exercise_data
6,FYR2 Banded Frog Pump,NaN,Abdominals,Bands,Intermediate,gym_exercise_data
7,Band low-to-high twist,The band low-to-high twist is a core exercise ...,Abdominals,Bands,Intermediate,gym_exercise_data
8,Barbell roll-out,The barbell roll-out is an abdominal exercise ...,Abdominals,Barbell,Intermediate,gym_exercise_data
9,Barbell Ab Rollout - On Knees,The barbell roll-out is an abdominal exercise ...,Abdominals,Barbell,Intermediate,gym_exercise_data


## Sauvegarde de df_all_unique en CSV

In [22]:
# Dossier de sortie (tu peux l’adapter à ta structure)
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)

# Nom du fichier avec timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
output_path = output_dir / f"dataset_exercices_fusion_{timestamp}.csv"

# Sauvegarde
df_all_unique.to_csv(output_path, index=False, encoding="utf-8")

print(f"✅ Fichier sauvegardé : {output_path}")

✅ Fichier sauvegardé : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\notebooks\dataset_fusion\outputs\dataset_exercices_fusion_20251127_2000.csv


## Sauvegarde de df_all_unique en JSON

In [23]:
# Dossier de sortie
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)

# Nom du fichier avec timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
output_path_json = output_dir / f"dataset_exercices_fusion_{timestamp}.json"

# Sauvegarde JSON (indenté & utf-8)
df_all_unique.to_json(
    output_path_json,
    orient="records",
    force_ascii=False,
    indent=4
)

print(f"✅ Fichier JSON sauvegardé : {output_path_json}")

✅ Fichier JSON sauvegardé : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\notebooks\dataset_fusion\outputs\dataset_exercices_fusion_20251127_2004.json
